In [1]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [2]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [3]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(temperature=0,model = "llama3-70b-8192")

In [5]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "data/Be_Good.pdf"

loader = PyPDFLoader(file_path)
docs=loader.load()
print(len(docs))

11


In [7]:
docs[0].page_content

"Be Good - Essay by Paul Graham\nBe Good\nBe good\nApril 2008(This essay is derived from a talk at the 2008 Startup School.)About a month after we\nstarted Y Combinator we came up with the\nphrase that became our motto: Make something people want.  We've\nlearned a lot since then, but if I were choosing now that's still\nthe one I'd pick.Another thing we tell founders is not to worry too much about the\nbusiness model, at least at first.  Not because making money is\nunimportant, but because it's so much easier than building something\ngreat.A couple weeks ago I realized that if you put those two ideas\ntogether, you get something surprising.  Make something people want.\nDon't worry too much about making money.  What you've got is a\ndescription of a charity.When you get an unexpected result like this, it could either be a\nbug or a new discovery.  Either businesses aren't supposed to be\nlike charities, and we've proven by reductio ad absurdum that one\nor both of the principles we b

In [9]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings

embedding_models = HuggingFaceEmbeddings()

d:\LLM Projects Basic\Chatbot with temporary memory\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
text_splitters = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitters.split_documents(docs)
vector_store = Chroma.from_documents(documents=splits, embedding=embedding_models)
retriever = vector_store.as_retriever()

In [11]:
# Constructing final RAG chain

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llmModel, prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

results = rag_chain.invoke({"input": "What is this article about?"})

results["answer"]

'This article, "Be Good" by Paul Graham, is about the importance of being benevolent and doing good for people, especially in the context of startups and business.'

In [12]:
results

{'input': 'What is this article about?',
 'context': [Document(id='2e19dc38-909f-4b32-a077-27a0c384635e', metadata={'author': 'Paul Graham', 'creationdate': 'D:20240613143635', 'creator': 'PyPDF', 'page': 10, 'page_label': '11', 'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'source': 'data/Be_Good.pdf', 'title': 'Be Good - Essay by Paul Graham', 'total_pages': 11}, page_content="Be Good - Essay by Paul Graham\nGoogle does.Most explicitly benevolent projects don't hold themselves sufficiently\naccountable.  They act as if having good intentions were enough to\nguarantee good effects.[3] Users dislike their\nnew operating system so much that they're starting petitions to\nsave the old one.  And the old one was nothing special.  The hackers\nwithin Microsoft must know in their hearts that if the company\nreally cared about users they'd just advise them to switch to OSX.Thanks to Trevor Blackwell, Paul\nBuchheit, Jessica Livingston,\nand Robert Morris for reading drafts of this